In [ ]:
!pip install ultralytics

In [ ]:
import os
import sys
import shutil
import json
import warnings
from pathlib import Path
from collections import Counter, defaultdict

import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split

import torch
from ultralytics import YOLO

from IPython.display import Image, display, clear_output

from tqdm import tqdm

warnings.filterwarnings('ignore')

In [ ]:
base = "/kaggle/input/competitions/dl-lab-2-stuff-detection/yolo_dataset/yolo_dataset"
train_images_dir = f"{base}/train/images"
train_labels_dir = f"{base}/train/labels"

add_images_dir = "/kaggle/input/datasets/iowiqo/staff-dop/add_images"
add_labels_dir = "/kaggle/input/datasets/iowiqo/staff-dop/add_labels"

output_dir = "/kaggle/working/staff_detection_dataset"

imgs_base = os.listdir(f"{base}/train/images")
imgs_add = os.listdir(add_images_dir)
all_imgs = imgs_base + [f"add_{f}" for f in imgs_add]

train, val = train_test_split(all_imgs, test_size=0.2, random_state=42)

for split, files in {"train": train, "val": val}.items():
    os.makedirs(f"{output_dir}/{split}/images", exist_ok=True)
    os.makedirs(f"{output_dir}/{split}/labels", exist_ok=True)
    
    for f in files:
        if f.startswith("add_"):
            img_src = f"{add_images_dir}/{f[4:]}"
            label_src = f"{add_labels_dir}/{f[4:].replace('.jpg','.txt')}"
        else:
            img_src = f"{base}/train/images/{f}"
            label_src = f"{base}/train/labels/{f.replace('.jpg','.txt')}"
            
        shutil.copy(img_src, f"{output_dir}/{split}/images/{os.path.basename(f)}")
        shutil.copy(label_src, f"{output_dir}/{split}/labels/{os.path.basename(f).replace('.jpg','.txt')}")

In [ ]:
with open(f"{output_dir}/data.yaml", "w") as f:
    f.write("""names:
  0: customer
  1: staff
path: /kaggle/working/staff_detection_dataset
train: train/images
val: val/images
""")

data = "/kaggle/working/staff_detection_dataset/data.yaml"

In [ ]:
def class_count(labels_dir):
    counter = Counter()
    for f in os.listdir(labels_dir):
        with open(os.path.join(labels_dir, f)) as file:
            for line in file:
                if line.strip():
                    counter[int(line.split()[0])] += 1
    return counter

train_counts = class_count(f'{output_dir}/train/labels')
val_counts = class_count(f'{output_dir}/val/labels')

print(f"Train: Customer: {train_counts.get(0, 0)}, Staff: {train_counts.get(1, 0)}")
print(f"Val:   Customer: {val_counts.get(0, 0)}, Staff: {val_counts.get(1, 0)}")

In [ ]:
model = YOLO("/kaggle/input/models/meowmeowmeeowww/y26m50ep/pytorch/default/1/y26m50ep.pt")

results = model.train(
    data=data,
    epochs=10,
    imgsz=1280,
    batch=-1,
    
    # Оптимизатор
    optimizer='MuSGD',
    lr0=0.001,
    lrf=0.01,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # Аугментация
    hsv_h=0.0,
    hsv_s=0.3,
    hsv_v=0.3,
    degrees=0.0,
    translate=0.1,
    scale=0.9,
    shear=0.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.2,
    cutmix=0.1,
    copy_paste=0.2,
    
    # Регуляризация
    dropout=0.1,
    weight_decay=0.0005,
    
    # Сохранение
    project='miet_yolo26',
    name='staff_detection',
    exist_ok=True,
    
    # Валидация
    val=True,
    plots=True,
    save=True,
    save_period=10,
    
    patience=0,
    
    conf=0.2,
    iou=0.4, 
    max_det=300,    
    single_cls=False,
)


results = model.train(
    data=data,
    epochs=20,
    imgsz=1280,
    batch=8,
    
    # Оптимизатор
    optimizer='MuSGD',
    lr0=0.0002,
    lrf=0.05,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # Аугментация
    hsv_h=0.0,
    hsv_s=0.2,
    hsv_v=0.2,
    degrees=0.0,
    translate=0.1,
    scale=0.9,
    shear=0.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    cutmix=0.1,
    copy_paste=0.1,
    
    # Регуляризация
    dropout=0.15,
    weight_decay=0.001,
    
    # Сохранение
    project='miet_yolo26',
    name='staff_detection',
    exist_ok=True,
    
    # Валидация
    val=True,
    plots=True,
    save=True,
    save_period=10,
    
    patience=0,
    
    conf=0.2,
    iou=0.4, 
    max_det=300,    
    single_cls=False,
    device='0,1'
)

In [ ]:
yolo11 = YOLO("yolo11m.pt")

results = yolo11.train(
    data=data,
    epochs=100,
    imgsz=1280,
    batch=10,
    
    # Оптимизатор
    optimizer='auto',
    lr0=0.0002,
    lrf=0.05,
    warmup_epochs=3,
    warmup_momentum=0.8,
    warmup_bias_lr=0.1,
    
    # Аугментация
    hsv_h=0.0,
    hsv_s=0.2,
    hsv_v=0.2,
    degrees=0.0,
    translate=0.1,
    scale=0.9,
    shear=0.0,
    perspective=0.0001,
    flipud=0.0,
    fliplr=0.5,
    mosaic=1.0,
    mixup=0.1,
    cutmix=0.1,
    copy_paste=0.1,
    
    # Регуляризация
    dropout=0.15,
    weight_decay=0.001,
    
    # Сохранение
    project='miet_yolo11',
    name='staff_detection',
    exist_ok=True,
    
    # Валидация
    val=True,
    plots=True,
    save=True,
    save_period=10,
    
    patience=30,
    
    conf=0.2,
    iou=0.4, 
    max_det=300,    
    single_cls=False,
    device='0,1'
)

In [ ]:
!pip install ensemble_boxes

In [ ]:
yolo11 = YOLO("/kaggle/input/models/meowmeowmeeowww/yolo11/pytorch/default/1/best_yolo11.pt")
yolo26 = YOLO("/kaggle/input/models/meowmeowmeeowww/80eps/pytorch/default/1/y26m80ep.pt")
predict_path = r'/kaggle/input/competitions/dl-lab-2-stuff-detection/test_images/test_images'

In [ ]:
res1 = yolo11.predict(predict_path, conf=0.01, iou=0.4, augment=True, classes=[1], verbose=False)
res2 = yolo26.predict(predict_path, conf=0.01, iou=0.6, classes=[1], verbose=False)

In [ ]:
from pathlib import Path
import numpy as np
from ensemble_boxes import weighted_boxes_fusion 

preds_dir = Path("preds")
preds_dir.mkdir(exist_ok=True)  

for r1, r2 in zip(res1, res2):
    img_stem = Path(r1.path).stem
    out_file = preds_dir / f"{img_stem}.txt"

    # Extract normalized boxes, scores, labels
    boxes = [r1.boxes.xyxyn.cpu().numpy(), r2.boxes.xyxyn.cpu().numpy()]
    scores = [r1.boxes.conf.cpu().numpy(), r2.boxes.conf.cpu().numpy()]
    labels = [r1.boxes.cls.cpu().numpy().astype(int), r2.boxes.cls.cpu().numpy().astype(int)]

    # Fuse with higher weight for the more accurate model
    f_boxes, f_scores, f_labels = weighted_boxes_fusion(
        boxes, scores, labels, weights=[1, 1.5], iou_thr=0.55
    )

    # If no detections, skip writing
    if len(f_boxes) == 0:
        continue

    # Convert from [x1, y1, x2, y2] (normalized) to [xc, yc, w, h] (normalized)
    x1 = f_boxes[:, 0]
    y1 = f_boxes[:, 1]
    x2 = f_boxes[:, 2]
    y2 = f_boxes[:, 3]

    xc = (x1 + x2) / 2.0
    yc = (y1 + y2) / 2.0
    w  = x2 - x1
    h  = y2 - y1

    # Write each detection as a line: class xc yc w h score
    with open(out_file, "w") as f:
        for cls, cx, cy, bw, bh, sc in zip(f_labels, xc, yc, w, h, f_scores):
            line = f"{cls} {cx:.6f} {cy:.6f} {bw:.6f} {bh:.6f} {sc:.6f}\n"
            f.write(line)

print(f"Predictions saved in {preds_dir}/")

In [ ]:
from pathlib import Path
import json
import pandas as pd


def build_submission_from_solution_order(
    solution_csv: str,
    preds_dir: str,
    output_csv: str = "submission.csv",
    image_col: str = "image_name",
    boxes_col: str = "boxes",
    row_id_col: str = "id",
    require_score: bool = True,
    clamp_score: bool = True,
    keep_only_class: int | None = None,  # например 1, если нужны только строки класса 1; None = все классы
) -> None:
    """
    Builds submission.csv in EXACTLY the same image_name order as solution.csv.

    - Reads solution_csv, takes image_name column order as ground truth order.
    - For each image_name, looks for preds_dir/<stem>.txt
      where stem is Path(image_name).stem
    - If missing -> boxes=[]
    - Prediction txt lines: class xc yc w h score
    - Output boxes: JSON [[xc,yc,w,h,score], ...]
    """
    sol_path = Path(solution_csv)
    pdir = Path(preds_dir)

    if not sol_path.exists():
        raise FileNotFoundError(f"solution_csv not found: {sol_path}")
    if not pdir.exists() or not pdir.is_dir():
        raise FileNotFoundError(f"preds_dir not found or not a dir: {pdir}")

    sol = pd.read_csv(sol_path)
    if image_col not in sol.columns:
        raise ValueError(f"solution.csv must contain column '{image_col}'")

    # Keep original order from solution
    image_names = sol[image_col].astype(str).tolist()

    rows = []
    for idx, image_name in enumerate(image_names):
        stem = Path(image_name).stem
        pred_file = pdir / f"{stem}.txt"

        boxes = []
        if pred_file.exists():
            content = pred_file.read_text(encoding="utf-8", errors="replace").strip()
            if content:
                for ln in content.splitlines():
                    ln = ln.strip()
                    if not ln:
                        continue
                    parts = ln.split()
                    if require_score and len(parts) < 6:
                        continue
                    if len(parts) < 5:
                        continue

                    try:
                        cls = int(float(parts[0]))
                        if keep_only_class is not None and cls != keep_only_class:
                            continue

                        xc = float(parts[1])
                        yc = float(parts[2])
                        w = float(parts[3])
                        h = float(parts[4])
                        sc = float(parts[5]) if len(parts) >= 6 else 1.0
                    except ValueError:
                        continue

                    if clamp_score:
                        sc = 0.0 if sc < 0.0 else (1.0 if sc > 0.7 else sc)

                    boxes.append([xc, yc, w, h, sc])

        rows.append(
            {
                row_id_col: idx,
                image_col: image_name,  # EXACT match
                boxes_col: json.dumps(boxes, separators=(",", ":")),
            }
        )

    sub = pd.DataFrame(rows, columns=[row_id_col, image_col, boxes_col])
    sub.to_csv(output_csv, index=False)
    print(f"Saved: {output_csv} ({len(sub)} rows). Missing preds filled with [] in solution order.")


# пример запуска:


In [ ]:
build_submission_from_solution_order(
    solution_csv=r"/kaggle/input/competitions/dl-lab-2-stuff-detection/sample_sub.csv",
    preds_dir=r"/kaggle/working/preds",
    output_csv="11_26wbf.csv",
    keep_only_class=1, 
    clamp_score = False
)